# 09 Rerank and analyze faithfulness

Select by AlignScore, then evaluate paired SummaC change as the primary endpoint. Report bootstrap intervals, metric correlations, and ROUGE-L quality changes.

**Kernel:** `.venv-scoring` (`.venv-scoring/Scripts/python.exe`).


In [2]:
from pathlib import Path
import sys
import os
import json
import pandas as pd
from IPython.display import display

ROOT = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / 'configs' / 'config.yaml').is_file()
             and (path / 'outputs' / 'baseline.csv').is_file()), None)
if ROOT is None:
    raise RuntimeError('Open this notebook from the csc173-midterm repository.')
os.chdir(ROOT)
if Path(sys.prefix).resolve() != (ROOT / '.venv-scoring').resolve():
    raise RuntimeError('Select the .venv-scoring notebook kernel, then restart.')
import hashlib

RUN = ROOT / 'outputs' / 'faithfulness' / 'run_01'
N = 618

print('Run folder:', RUN)
print('Kernel:', sys.executable)


Run folder: c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\outputs\faithfulness\run_01
Kernel: c:\Users\Ramski\Documents\VSCode\Projects\csc173-midterm\.venv-scoring\Scripts\python.exe


## Function definitions


### Shared helper functions

Reading, validation, file saving, and resumption helpers.


In [3]:
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def read_csv(path):
    return pd.read_csv(path, dtype=str, keep_default_na=False, encoding='utf-8-sig')


def require(condition, message):
    if not condition:
        raise ValueError(message)


def nonblank(frame, columns):
    for column in columns:
        require(column in frame, f'Missing column: {column}')
        require(frame[column].map(lambda x: isinstance(x, str) and bool(x.strip())).all(),
                f'Blank or invalid text in {column}')


def reference_list(text):
    values = json.loads(text)
    require(isinstance(values, list) and len(values) > 0, 'Missing reference list')
    require(all(isinstance(x, str) and bool(x.strip()) for x in values), 'Invalid reference')
    return values


def atomic_csv(frame, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    frame.to_csv(temporary, index=False, lineterminator='\n')
    os.replace(temporary, path)


def write_json(value, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                    allow_nan=False), encoding='utf-8')
    os.replace(temporary, path)


def manifest(path, value):
    # Reject changed inputs/configuration instead of reusing incompatible partial results.
    encoded = json.dumps(value, sort_keys=True, ensure_ascii=False, allow_nan=False)
    if path.exists():
        old = json.dumps(json.loads(path.read_text(encoding='utf-8')),
                         sort_keys=True, ensure_ascii=False, allow_nan=False)
        require(old == encoded, f'Protocol changed: {path}. Use a new run folder.')
    else:
        write_json(value, path)


def load_eval():
    frame = read_csv(RUN / 'evaluation_pairs.csv')
    nonblank(frame, ['sample_id', 'abstract', 'baseline_summary', 'human_tldrs'])
    require(len(frame) == N and frame.sample_id.is_unique, 'Expected 618 unique papers')
    frame.human_tldrs.map(reference_list)
    return frame


def validate_candidates(frame, evaluation):
    nonblank(frame, ['sample_id', 'candidate_id', 'abstract', 'candidate_summary'])
    require(len(frame) == N * 5, 'Expected 3090 candidate rows')
    require(set(frame.sample_id) == set(evaluation.sample_id), 'Candidate ID set differs')
    require(not frame.duplicated(['sample_id', 'candidate_id']).any(), 'Duplicate keys')
    for sample_id, group in frame.groupby('sample_id', sort=False):
        require(set(group.candidate_id.astype(str)) == set(map(str, range(5))),
                f'Candidate IDs must be 0..4: {sample_id}')
        normalized = group.candidate_summary.map(lambda text: ' '.join(text.split()))
        require(normalized.nunique() == 5, f'Non-distinct candidates: {sample_id}')
    joined = frame.merge(evaluation[['sample_id', 'abstract']], on='sample_id',
                         validate='many_to_one', suffixes=('', '_expected'))
    require((joined.abstract == joined.abstract_expected).all(), 'Wrong candidate source')


### `analyze_reranking()`

Choose winners by AlignScore, pair them with baseline summaries, and compute statistics and plots.

#### Inputs and purpose

This function joins the completed baseline and candidate evaluations into a paper-level experiment. It reads evaluation pairs, both score tables, and generated candidates. `numeric_scores()` converts CSV score strings into numbers and rejects invalid values. Further checks ensure complete IDs, matching text, and compatible input/settings manifests.

The nested helpers calculate statistics for this analysis; defining them does not run those calculations until they are called later.

#### Select one winner per paper

Candidate IDs become integers. Sorting by `sample_id`, descending `alignscore_score`, and ascending `candidate_id` places the winning row first for each paper. `drop_duplicates('sample_id')` keeps it. Exact AlignScore ties use the smaller candidate ID; SummaC never participates in selection or tie-breaking.

For example, candidates 1 and 2 both scoring AlignScore 0.88 lead to candidate 1 winning, even if candidate 2 has higher SummaC. The selected rows are reordered to match evaluation IDs and saved as `reranked_summaries.csv`.

#### Pair winners with their own baselines

Baseline and selected score columns receive different names, then a one-to-one merge on `sample_id` matches each paper's baseline to that paper's winner. This avoids comparing unrelated rows merely because their positions happen to match.

For each faithfulness metric, delta equals selected minus baseline. Baseline SummaC 0.60 and selected 0.75 produces +0.15; selected 0.55 produces ?0.05. Positive means improvement according to that metric, not guaranteed human-verified factual improvement.

SummaC change is the primary endpoint because AlignScore already chose winners. Its gain is selection-favored. SummaC is separate from the selection rule, though the metrics may correlate and share weaknesses.

#### Reference-based quality check

ROUGE-L uses longest-common-subsequence overlap with human TLDRs, with stemming enabled. For each summary, the function evaluates every reference and takes the maximum F1. Scores of 0.20, 0.35, and 0.28 therefore give that summary 0.35. The same rule applies to baseline and selected summaries.

ROUGE-L measures reference overlap, not factual support. A faithful paraphrase can have low overlap; an unsupported statement can reuse reference words. `delta_rougeL` is selected minus baseline ROUGE-L.

#### Descriptive statistics and metric agreement

`describe()` calculates mean, sample standard deviation (`ddof=1`, denominator n?1), minimum, and maximum for SummaC and AlignScore. These describe baseline and selected groups separately, not all candidates pooled together. Standard deviation is score spread across papers, not uncertainty in the mean change.

`correlation()` calculates Pearson linear association and Spearman rank association between the two metrics. Positive association means higher scores on one tend to accompany higher scores on the other; it does not establish agreement with human factual judgments. The saved correlation p-values do not test the primary before/after question. If either metric is constant, correlations are undefined and written as JSON `null` with a reason.

#### Paired bootstrap uncertainty

`paired_summary()` receives 618 paper-level deltas. It samples 618 deltas with replacement, calculates their mean, and repeats this 10,000 times using seed 42. Some papers repeat in each draw and others are omitted. The 2.5th and 97.5th percentiles of the resampled means form the 95% percentile bootstrap interval.

Resampling deltas preserves the before/after pairing. The five candidates from a paper are not treated as independent papers. The interval concerns the mean change, not the range of changes for individual papers.

Changes larger than `1e-8` count as improved, smaller than `-1e-8` as declined, and the rest as unchanged. This tolerance affects counts only; raw deltas still determine means and bootstrap intervals. The helper runs separately for SummaC, AlignScore, and ROUGE-L.

#### What is written

`reranking_comparison.csv` contains paired text, scores, ROUGE-L values, and deltas. `baseline_statistics.json` holds baseline statistics, while `evaluation_results.json` holds the main report. The two-panel `metric_scatter.png` plots SummaC versus AlignScore for baseline and selected summaries; it is an association plot, not a direct plot of paired gains.

Automatic metrics do not prove factual correctness. No non-inferiority margin or test is implemented, so a ROUGE interval crossing zero does not establish that quality was preserved.



In [10]:
def analyze_reranking():
    """Select by AlignScore and compare winners against the paired baseline."""
    import json
    import numpy as np
    import pandas as pd
    from scipy import stats
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    from rouge_score import rouge_scorer

    def numeric_scores(frame):
        for name in ['summac_score', 'alignscore_score']:
            frame[name] = pd.to_numeric(frame[name], errors='raise')
            values = frame[name].to_numpy()
            require(np.isfinite(values).all() and ((values >= 0) & (values <= 1)).all(),
                    f'Invalid {name}')
        return frame

    def describe(frame):
        return {name: {'mean': float(frame[name].mean()),
                       'sd_sample': float(frame[name].std(ddof=1)),
                       'min': float(frame[name].min()), 'max': float(frame[name].max())}
                for name in ['summac_score', 'alignscore_score']}

    def correlation(frame):
        x, y = frame.summac_score.to_numpy(), frame.alignscore_score.to_numpy()
        if np.ptp(x) == 0 or np.ptp(y) == 0:
            return {'n': len(frame), 'pearson_r': None, 'spearman_rho': None,
                    'reason': 'Correlation undefined for a constant metric'}
        pearson, spearman = stats.pearsonr(x, y), stats.spearmanr(x, y)
        return {'n': len(frame), 'pearson_r': float(pearson.statistic),
                'pearson_p_two_sided': float(pearson.pvalue),
                'spearman_rho': float(spearman.statistic),
                'spearman_p_two_sided_asymptotic': float(spearman.pvalue)}

    def paired_summary(delta, seed=42, repeats=10000):
        delta = np.asarray(delta, dtype=float)
        require(len(delta) == N and np.isfinite(delta).all(), 'Need 618 paired deltas')
        rng = np.random.default_rng(seed)
        means = np.empty(repeats)
        for index in range(repeats):
            means[index] = rng.choice(delta, size=len(delta), replace=True).mean()
        low, high = np.quantile(means, [.025, .975])
        # Predeclared practical numerical tolerance for the unchanged count only.
        improved = int((delta > 1e-8).sum())
        declined = int((delta < -1e-8).sum())
        unchanged = len(delta) - improved - declined
        return {'mean_delta': float(delta.mean()), 'ci95_percentile': [float(low), float(high)],
                'n': len(delta), 'bootstrap_repeats': repeats, 'seed': seed,
                'unchanged_tolerance': 1e-8, 'improved': improved, 'declined': declined,
                'unchanged': unchanged,
                'improved_pct': 100 * improved / len(delta),
                'declined_pct': 100 * declined / len(delta),
                'unchanged_pct': 100 * unchanged / len(delta)}

    evaluation = load_eval()
    baseline = numeric_scores(read_csv(RUN / 'baseline_scores.csv'))
    candidates = numeric_scores(read_csv(RUN / 'candidate_scores.csv'))
    generated = read_csv(RUN / 'generated_candidates.csv')
    validate_candidates(candidates, evaluation)
    validate_candidates(generated, evaluation)
    require(len(baseline) == N and baseline.sample_id.is_unique, 'Incomplete baseline scores')
    require(set(baseline.sample_id) == set(evaluation.sample_id), 'Baseline IDs differ')
    baseline = baseline.set_index('sample_id').loc[evaluation.sample_id].reset_index()
    for column in ['abstract', 'baseline_summary', 'human_tldrs']:
        require(baseline[column].tolist() == evaluation[column].tolist(), 'Baseline text differs')
    keys = ['sample_id', 'candidate_id']
    check = candidates.merge(generated, on=keys, validate='one_to_one', suffixes=('_s', '_g'))
    require(len(check) == N * 5, 'Candidate score/input key mismatch')
    for column in ['abstract', 'candidate_summary', 'raw_rank']:
        require((check[column + '_s'] == check[column + '_g']).all(), 'Candidate text/metadata differs')

    # Verify scores are bound to current inputs and the same metric manifest.
    metric_hash = sha256(RUN / 'metric_manifest.json')
    for kind, input_name in [('baseline', 'evaluation_pairs.csv'),
                             ('candidate', 'generated_candidates.csv')]:
        saved = json.loads((RUN / f'{kind}_score_manifest.json').read_text(encoding='utf-8'))
        print(f"Saved:  {saved}")
        print("metric_hash: " + metric_hash)
        require(saved == {'input_sha256': sha256(RUN / input_name),
                         'metric_manifest_sha256': metric_hash}, 'Scoring provenance mismatch')

    manifest(RUN / 'analysis_manifest.json', {
        'baseline_scores_sha256': sha256(RUN / 'baseline_scores.csv'),
        'candidate_scores_sha256': sha256(RUN / 'candidate_scores.csv'),
        'tie_rule': 'exact AlignScore tie -> smallest candidate ID',
        'primary_endpoint': 'mean paired SummaC change', 'bootstrap_repeats': 10000,
        'seed': 42, 'unchanged_tolerance': 1e-8,
        'rouge': 'rougeL F1, stemming, maximum reference F1 per paper',
    })
    candidates['candidate_id'] = candidates.candidate_id.astype(int)
    ranked = candidates.sort_values(['sample_id', 'alignscore_score', 'candidate_id'],
                                   ascending=[True, False, True], kind='stable')
    winners = ranked.drop_duplicates('sample_id').copy()
    winners = winners.set_index('sample_id').loc[evaluation.sample_id].reset_index()
    require(len(winners) == N, 'Expected 618 winners')
    atomic_csv(winners, RUN / 'reranked_summaries.csv')

    paired = baseline.rename(columns={'summac_score': 'baseline_summac',
                                      'alignscore_score': 'baseline_alignscore'})
    right = winners[['sample_id', 'candidate_id', 'candidate_summary',
                     'summac_score', 'alignscore_score']].rename(columns={
        'candidate_id': 'selected_candidate_id', 'candidate_summary': 'reranked_summary',
        'summac_score': 'reranked_summac', 'alignscore_score': 'reranked_alignscore'})
    paired = paired.merge(right, on='sample_id', validate='one_to_one')
    for metric in ['summac', 'alignscore']:
        paired[f'delta_{metric}'] = paired[f'reranked_{metric}'] - paired[f'baseline_{metric}']

    rouge = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    def reference_rouge(summary, references):
        return max(rouge.score(reference, summary)['rougeL'].fmeasure for reference in references)
    refs = paired.human_tldrs.map(reference_list)
    paired['baseline_rougeL'] = [reference_rouge(s, r) for s, r in zip(paired.baseline_summary, refs)]
    paired['reranked_rougeL'] = [reference_rouge(s, r) for s, r in zip(paired.reranked_summary, refs)]
    paired['delta_rougeL'] = paired.reranked_rougeL - paired.baseline_rougeL
    atomic_csv(paired, RUN / 'reranking_comparison.csv')

    baseline_stats = {'n': N, 'metrics': describe(baseline), 'correlation': correlation(baseline)}
    write_json(baseline_stats, RUN / 'baseline_statistics.json')
    results = {'n_papers': N, 'n_candidates': len(candidates),
        'baseline': baseline_stats,
        'reranked': {'metrics': describe(winners), 'correlation': correlation(winners)},
        'paired': {metric: paired_summary(paired[f'delta_{metric}'])
                   for metric in ['summac', 'alignscore', 'rougeL']},
        'rougeL_means': {'baseline': float(paired.baseline_rougeL.mean()),
                         'reranked': float(paired.reranked_rougeL.mean())},
        'primary_endpoint': 'summac',
        'notes': ['Automatic metric changes do not prove factual correctness.',
                  'AlignScore selected winners; its gain is selection-favored.',
                  'ROUGE CI including zero does not establish non-inferiority.']}
    write_json(results, RUN / 'evaluation_results.json')

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    for axis, data, title in zip(axes, [baseline, winners], ['Baseline', 'Selected']):
        axis.scatter(data.summac_score, data.alignscore_score, s=10, alpha=.5)
        axis.set(xlabel='SummaCZS entailment-only', ylabel='AlignScore-base',
                 title=title, xlim=(0, 1), ylim=(0, 1))
    fig.tight_layout()
    fig.savefig(RUN / 'metric_scatter.png', dpi=180)
    plt.close(fig)
    print(json.dumps(results, indent=2, allow_nan=False))


## Execution and results


## 10. Select, compare, and report

Primary endpoint: mean paired SummaC change. AlignScore gains are selection-favored.
ROUGE-L uses the maximum reference F1 per paper and is a quality check, not a faithfulness
proof. A quality-change interval crossing zero does not establish non-inferiority.
`ci95_percentile[0]` and `[1]` are bounds on mean change, not individual-score extremes.

#### Reading the mean changes and confidence intervals

The displayed table summarizes one paper-level delta per metric. A positive `mean_change` means the selected summaries have a higher average score than the baselines. For example, +0.03 is three hundredths on that metric's scale; it is not automatically a 3% reduction in hallucinations.

`CI_low` and `CI_high` are the bootstrap interval bounds for mean change. An illustrative interval `[0.02, 0.06]` supports positive mean change under this procedure. `[-0.01, 0.04]` includes zero, so the change is not clearly separated from zero by this interval criterion. These bounds are not the minimum/maximum changes across papers and do not mean that 95% of papers improved.

The `improved`, `declined`, and `unchanged` columns count papers using the numerical tolerance described above. They answer a different question from the mean: a few large gains can coexist with many small declines. Inspect both the average and counts.

#### Why the metrics have different roles

SummaC is the predeclared primary endpoint. AlignScore is useful to report, but its gains must be described as selection-favored because the algorithm explicitly chooses its highest-scoring candidate. Neither score is a substitute for reading examples or human factuality assessment.

ROUGE-L is an additional quality/overlap check against human references. An interval including zero is not evidence of equivalence or non-inferiority. Those conclusions require an explicitly defined tolerance and an appropriate analysis; they are not implemented here.

#### Correlation table and figure

The correlation display compares how strongly the two faithfulness metrics agree across baseline papers and selected papers. It does not measure the correlation between a baseline and its own selected summary. The scatter plot shows the same two metric axes in separate baseline/selected panels; each point is one paper.

All numbers shown come from saved `evaluation_results.json` after the analysis call. Interpret them together with their inputs and the paired comparison CSV, rather than treating an old displayed notebook output as proof of the latest run.


In [11]:
analyze_reranking()

results_path = RUN / 'evaluation_results.json'
if results_path.exists():
    results = json.loads(results_path.read_text(encoding='utf-8'))
    rows = []
    for metric in ['summac', 'alignscore', 'rougeL']:
        paired = results['paired'][metric]
        rows.append({'metric': metric, 'mean_change': paired['mean_delta'],
                     'CI_low': paired['ci95_percentile'][0],
                     'CI_high': paired['ci95_percentile'][1],
                     'improved': paired['improved'], 'declined': paired['declined'],
                     'unchanged': paired['unchanged']})
    display(pd.DataFrame(rows).set_index('metric'))
    display(pd.DataFrame({'baseline': results['baseline']['correlation'],
                          'selected': results['reranked']['correlation']}))
    from IPython.display import Image
    display(Image(filename=str(RUN / 'metric_scatter.png')))
else:
    print('No completed analysis results yet.')


Saved:  {'input_sha256': '6cef68233c8efe75e86df6bcfce86a45ab41befd43d6a27a4cd12287c2209d1b', 'metric_manifest_sha256': '7434f120546315ea4162fc116214f541300672d659571e769d0bb7025368fe3f'}
metric_hash: 9fe4769e0ad712d84aac1d3b9333def828ae2989e79bc6b462a38de9d1c17375


ValueError: Scoring provenance mismatch

## 11. Inspect examples and artifacts

`expected` maps output names to required counts. A count alone does not prove correctness;
the stage validators additionally verify IDs, text, score ranges, and manifests.
The final message identifies the derived-output folder; it does not perform a baseline hash comparison.

#### Inspecting concrete examples

The preview reads `reranking_comparison.csv` and displays baseline and selected text with the two faithfulness deltas. These are the first rows in the saved table, not a random or representative human evaluation sample. Reading them can reveal what a positive or negative score change looks like, but a few examples cannot establish dataset-wide factual accuracy.

Each row is a matched paper: compare both summaries against that paper's abstract in the same saved table. A metric improvement is informative, but unsupported details can still be present.

#### Understanding the completeness table

`expected` maps each filename to its complete row count. The loop reads each existing file, counts its rows, and records one status:

| Status | Meaning |
|---|---|
| `missing` | No file exists at the expected path |
| `partial/invalid` | Row count differs from the declared full-study count |
| `row count complete; inspect validation` | Count matches, but content correctness is not established by this table |

Expected totals follow the experiment design: 618 evaluation rows, 618 baseline score rows, 6,180 raw returns, 3,090 retained candidates and candidate score rows, then 618 winners and comparison rows. Raw generation contains ten returns per paper; final candidate files contain only five retained distinct summaries per paper.

This display is a quick inventory. It does not itself rerun every ID, text, distinctness, numeric-score, or manifest check, and a same-sized file could still contain incorrect data. Those checks occur in the stage functions.

#### About the final baseline message

The code below prints that the original baseline is read only and identifies `RUN`. It does **not** calculate a SHA-256 equality check against a frozen baseline hash. That statement describes the workflow's write locations; it is not a fresh file-integrity test. Derived results are written under `RUN`, while `outputs/baseline.csv` is used as input.


In [5]:
comparison_path = RUN / 'reranking_comparison.csv'
if comparison_path.exists():
    comparison = read_csv(comparison_path)
    display(comparison[['sample_id', 'baseline_summary', 'reranked_summary',
                        'delta_summac', 'delta_alignscore']].head())
expected = {'evaluation_pairs.csv': 618, 'baseline_scores.csv': 618,
            'generation_pool.csv': 6180, 'generated_candidates.csv': 3090,
            'candidate_scores.csv': 3090, 'reranked_summaries.csv': 618,
            'reranking_comparison.csv': 618}
status = []
for name, count in expected.items():
    path = RUN / name
    actual = len(read_csv(path)) if path.exists() else None
    status.append({'file': name, 'expected_rows': count, 'saved_rows': actual,
                   'status': 'missing' if actual is None else
                             'row count complete; inspect validation' if actual == count else 'partial/invalid'})
display(pd.DataFrame(status))
print('Original baseline is read only; derived tables are saved in:', RUN)


,sample_id,baseline_summary,reranked_summary,delta_summac,delta_alignscore
0,SJ1Xmf-Rb,"We propose FearNet, a generative model that do...",We propose FearNet for incremental class learn...,-0.67584228515625,0.0013111233711242676
1,S1xzyhR9Y7,Multi-view learning can provide self-supervisi...,Multi-view learning can provide self-supervisi...,-0.00537109375,0.0019704103469848633
2,HJDUjKeA-,We show how discrete objects can be learnt in ...,We construct a differentiable mapping from an ...,-0.92962646484375,0.004933536052703857
3,BJgLg3R9KQ,We extend a state-of-the-art attention network...,We show the benefit of using stronger supervis...,0.885986328125,0.003973484039306641
4,BklpOo09tQ,We show that adversarial examples can be gener...,We show that adversarial examples can be gener...,-0.0029296875,0.010419726371765137


,file,expected_rows,saved_rows,status
0,evaluation_pairs.csv,618,618,row count complete; inspect validation
1,baseline_scores.csv,618,618,row count complete; inspect validation
2,generation_pool.csv,6180,6180,row count complete; inspect validation
3,generated_candidates.csv,3090,3090,row count complete; inspect validation
4,candidate_scores.csv,3090,3090,row count complete; inspect validation
5,reranked_summaries.csv,618,618,row count complete; inspect validation
6,reranking_comparison.csv,618,618,row count complete; inspect validation


Original baseline is read only; derived tables are saved in: c:\Users\Yelfia\Documents\DeepLearning\MIDTERMPROJ\csc173-midterm\outputs\faithfulness\run_01
